In [1]:

# 0. INSTALL DEPENDENCIES
%pip install flask flask-cors pyngrok imbalanced-learn scikit-learn pandas numpy matplotlib seaborn scipy -q


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:

import os, sys, time, warnings, csv, threading, pickle, io, base64
from datetime import datetime
from math import gamma as G

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.svm             import SVC
from sklearn.preprocessing   import StandardScaler
from sklearn.impute          import SimpleImputer
from sklearn.pipeline        import Pipeline
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.metrics         import (accuracy_score, classification_report,
                                     confusion_matrix, ConfusionMatrixDisplay,
                                     roc_auc_score, roc_curve,
                                     precision_recall_curve,
                                     average_precision_score,
                                     f1_score, matthews_corrcoef)

from flask import Flask, request, jsonify, render_template_string, send_file
from flask_cors import CORS

warnings.filterwarnings('ignore')
np.random.seed(2024)
print('Imports OK')


Imports OK


## Section 1 — Data Loading & Preprocessing

In [3]:
# 1. DATA LOADING & PREPROCESSING

COLS = ['age','sex','cp','trestbps','chol','fbs','restecg',
        'thalach','exang','oldpeak','slope','ca','thal','target']

DATA_FILES = [
    os.path.expanduser('~/Downloads/heart+disease/processed.cleveland.data'),
    os.path.expanduser('~/Downloads/heart+disease/processed.hungarian.data'),
    os.path.expanduser('~/Downloads/heart+disease/processed.va.data'),
    os.path.expanduser('~/Downloads/heart+disease/processed.switzerland.data'),
]

frames = []
for f in DATA_FILES:
    df = pd.read_csv(f, header=None, names=COLS, na_values='?')
    frames.append(df)

data = pd.concat(frames, ignore_index=True)
print(f"[DATA] Raw combined shape: {data.shape}")

data['target'] = (data['target'] > 0).astype(int)

drop_c = data.isnull().mean()[lambda s: s > 0.40].index.tolist()
data.drop(columns=drop_c, inplace=True)
data_arr = SimpleImputer(strategy='median').fit_transform(data)
data = pd.DataFrame(data_arr, columns=data.columns)

FEATURES = [c for c in data.columns if c != 'target']
X_raw = data[FEATURES].values.astype(np.float64)
y     = data['target'].values.astype(int)
n_feat = len(FEATURES)

print(f"  Samples  : {len(y)}  |  Disease={y.sum()}  Healthy={len(y)-y.sum()}")
print(f"  Features : {n_feat} → {FEATURES}")

[DATA] Raw combined shape: (920, 14)
  Samples  : 920  |  Disease=509  Healthy=411
  Features : 11 → ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope']


## Section 1.5 — Missing Values & Class Balancing (SMOTE)

In [4]:
# 1.5 MISSING VALUES & DATA BALANCING ANALYSIS
print("\n" + "═"*70)
print("  MISSING VALUES & CLASS IMBALANCE ANALYSIS")
print("═"*70)

print("\n  [MISSING] Before preprocessing:")
missing_before = data.isnull().sum()
print(f"  Total missing values: {missing_before.sum()}")
if missing_before.sum() > 0:
    print("  Missing by column:")
    print(missing_before[missing_before > 0])

print("\n  [IMBALANCE] Target distribution (BEFORE SMOTE):")
unique, counts = np.unique(y, return_counts=True)
for u, c in zip(unique, counts):
    label = 'Disease' if u == 1 else 'No Disease'
    print(f"    {label:<15}: {c:4d} ({c/len(y)*100:5.1f}%)")
imbalance_ratio = counts.max() / counts.min()
print(f"  Imbalance Ratio: {imbalance_ratio:.2f}:1")

print("\n  [SMOTE] Applying SMOTE for balancing …")
try:
    from imblearn.over_sampling import SMOTE
    smote = SMOTE(random_state=42, k_neighbors=3)
    X_balanced, y_balanced = smote.fit_resample(X_raw, y)
    print(f"  ✔  SMOTE applied successfully!")
    print(f"  Original shape: {X_raw.shape}")
    print(f"  Balanced shape: {X_balanced.shape}")
    print("\n  [IMBALANCE] Target distribution (AFTER SMOTE):")
    unique_bal, counts_bal = np.unique(y_balanced, return_counts=True)
    for u, c in zip(unique_bal, counts_bal):
        label = 'Disease' if u == 1 else 'No Disease'
        print(f"    {label:<15}: {c:4d} ({c/len(y_balanced)*100:5.1f}%)")
    print(f"  New Imbalance Ratio: {counts_bal.max() / counts_bal.min():.2f}:1")
    X_raw = X_balanced
    y = y_balanced
except ImportError:
    print("  ⚠  imbalanced-learn not installed. Using original data without SMOTE.")


══════════════════════════════════════════════════════════════════════
  MISSING VALUES & CLASS IMBALANCE ANALYSIS
══════════════════════════════════════════════════════════════════════

  [MISSING] Before preprocessing:
  Total missing values: 0

  [IMBALANCE] Target distribution (BEFORE SMOTE):
    No Disease     :  411 ( 44.7%)
    Disease        :  509 ( 55.3%)
  Imbalance Ratio: 1.24:1

  [SMOTE] Applying SMOTE for balancing …
  ✔  SMOTE applied successfully!
  Original shape: (920, 11)
  Balanced shape: (1018, 11)

  [IMBALANCE] Target distribution (AFTER SMOTE):
    No Disease     :  509 ( 50.0%)
    Disease        :  509 ( 50.0%)
  New Imbalance Ratio: 1.00:1


## Section 2 — Deep Drizzle Optimization Algorithm (Class Definition)

In [5]:

# 2. DEEP DRIZZLE OPTIMIZATION ALGORITHM
class DeepDOA:
    """
    Each droplet encodes a JOINT solution:
        x = [ w_1, ..., w_n, log10(C), log10(gamma) ]

    Mechanisms
    ──────────
    1. Cosine-annealed evaporation  α_t : 0.95 → 0.40
    2. Lévy-flight drizzle noise    (heavy-tail exploration)
    3. Adaptive β_local / β_global  (grow local, shrink global over time)
    4. Elite condensation           (worst 20% → re-init near top-5 cloud)
    5. Gravity surge                (burst toward global best every surge_every)
    6. Growing neighbourhood size   (nb_init → nb_max over iterations)
    7. Elitist lock                 (top elite_k preserved each round)
    """

    @staticmethod
    def _levy(dim, scale, beta=1.5):
        sig = (G(1+beta)*np.sin(np.pi*beta/2) /
               (G((1+beta)/2)*beta*2**((beta-1)/2)))**(1/beta)
        u = np.random.normal(0, sig, dim)
        v = np.abs(np.random.normal(0, 1, dim))
        return scale * u / (v**(1/beta))

    def __init__(self,
                 n_droplets=25, n_iter=120,
                 w_min=0.001, w_max=5.0,
                 logC_min=-1.0, logC_max=3.0,
                 logg_min=-3.0, logg_max=1.0,
                 alpha_max=0.95, alpha_min=0.40,
                 bl_init=0.30, bl_end=0.60,
                 bg_init=0.65, bg_end=0.25,
                 levy_init=0.10, levy_min=0.003,
                 cond_pct=0.20, cond_sigma=0.15,
                 surge_every=15, surge_str=0.65,
                 nb_init=4, nb_max=12,
                 elite_k=3, cv_folds=5,
                 verbose=True):
        self.n_droplets = n_droplets; self.n_iter = n_iter
        self.w_min = w_min; self.w_max = w_max
        self.logC_min = logC_min; self.logC_max = logC_max
        self.logg_min = logg_min; self.logg_max = logg_max
        self.alpha_max = alpha_max; self.alpha_min = alpha_min
        self.bl_init = bl_init; self.bl_end = bl_end
        self.bg_init = bg_init; self.bg_end = bg_end
        self.levy_init = levy_init; self.levy_min = levy_min
        self.cond_pct = cond_pct; self.cond_sigma = cond_sigma
        self.surge_every = surge_every; self.surge_str = surge_str
        self.nb_init = nb_init; self.nb_max = nb_max
        self.elite_k = elite_k; self.cv_folds = cv_folds
        self.verbose = verbose
        self.hist_best = []; self.hist_mean = []; self.hist_std = []
        self.hist_C = []; self.hist_gam = []; self.hist_weights = []
        self.alpha_hist = []; self.bl_hist = []; self.bg_hist = []; self.levy_hist = []

    def _lb(self, D):
        return np.concatenate([np.full(D-2, self.w_min), [self.logC_min, self.logg_min]])

    def _ub(self, D):
        return np.concatenate([np.full(D-2, self.w_max), [self.logC_max, self.logg_max]])

    def _decode(self, x):
        return x[:-2], 10**x[-2], 10**x[-1]

    def _fitness(self, x, X, y):
        w, C, g = self._decode(x)
        pipe = Pipeline([
            ('sc', StandardScaler()),
            ('sv', SVC(C=C, kernel='rbf', gamma=g, probability=False, random_state=42))
        ])
        kf = StratifiedKFold(n_splits=self.cv_folds, shuffle=True, random_state=42)
        return cross_val_score(pipe, X * w, y, cv=kf, scoring='accuracy', n_jobs=1).mean()

    def _cosine(self, t, hi, lo):
        return lo + 0.5*(hi - lo)*(1 + np.cos(np.pi * t / self.n_iter))

    def optimize(self, X, y):
        D = X.shape[1] + 2
        lb = self._lb(D); ub = self._ub(D)
        pos = np.random.uniform(lb, ub, (self.n_droplets, D))
        vel = np.random.uniform(-0.05, 0.05, (self.n_droplets, D))
        fit = np.array([self._fitness(pos[i], X, y) for i in range(self.n_droplets)])
        pb_pos = pos.copy(); pb_fit = fit.copy()
        gb_idx = np.argmax(fit)
        gb_pos = pos[gb_idx].copy(); gb_fit = fit[gb_idx]
        w0, C0, g0 = self._decode(gb_pos)
        self.hist_best.append(gb_fit); self.hist_mean.append(fit.mean()); self.hist_std.append(fit.std())
        self.hist_C.append(C0); self.hist_gam.append(g0)
        if self.verbose:
            print(f"  Init    | best={gb_fit:.4f} | C={C0:.4f} | γ={g0:.6f}")
        for t in range(1, self.n_iter + 1):
            p = t / self.n_iter
            α = self._cosine(t, self.alpha_max, self.alpha_min)
            β_l = self.bl_init + (self.bl_end - self.bl_init) * p
            β_g = self.bg_init + (self.bg_end - self.bg_init) * p
            lev = max(self.levy_min, self.levy_init * (1 - 0.88 * p))
            nb = min(self.nb_max, self.nb_init + int((self.nb_max - self.nb_init) * p))
            surge = (t % self.surge_every == 0)
            self.alpha_hist.append(α); self.bl_hist.append(β_l)
            self.bg_hist.append(β_g); self.levy_hist.append(lev)
            elite_set = set(np.argsort(fit)[-self.elite_k:])
            n_cond = max(1, int(self.cond_pct * self.n_droplets))
            worst_idx = np.argsort(fit)[:n_cond]
            top5_idx = np.argsort(fit)[-min(5, self.n_droplets):]
            for ci in worst_idx:
                if ci in elite_set: continue
                seed = pos[np.random.choice(top5_idx)]
                noise = np.random.normal(0, self.cond_sigma, D)
                pos[ci] = np.clip(seed + noise, lb, ub)
                vel[ci] = np.random.uniform(-0.03, 0.03, D)
                fit[ci] = self._fitness(pos[ci], X, y)
                if fit[ci] > pb_fit[ci]: pb_fit[ci] = fit[ci]; pb_pos[ci] = pos[ci].copy()
                if fit[ci] > gb_fit: gb_fit = fit[ci]; gb_pos = pos[ci].copy()
            snap = []
            for i in range(self.n_droplets):
                if i in elite_set: snap.append(fit[i]); continue
                avail = [j for j in range(self.n_droplets) if j != i]
                nb_sel = np.random.choice(avail, nb, replace=False)
                nb_pos = pos[nb_sel[np.argmax(fit[nb_sel])]]
                r1, r2 = np.random.rand(D), np.random.rand(D)
                vel[i] = (α * vel[i] + β_l * r1 * (nb_pos - pos[i])
                          + β_g * r2 * (gb_pos - pos[i]) + self._levy(D, lev))
                if surge:
                    r3 = np.random.rand(D)
                    vel[i] += self.surge_str * r3 * (gb_pos - pos[i])
                pos[i] = np.clip(pos[i] + vel[i], lb, ub)
                fit[i] = self._fitness(pos[i], X, y)
                snap.append(fit[i])
                if fit[i] > pb_fit[i]: pb_fit[i] = fit[i]; pb_pos[i] = pos[i].copy()
                if fit[i] > gb_fit: gb_fit = fit[i]; gb_pos = pos[i].copy()
            self.hist_best.append(gb_fit); self.hist_mean.append(np.mean(snap)); self.hist_std.append(np.std(snap))
            w_t, C_t, g_t = self._decode(gb_pos)
            self.hist_C.append(C_t); self.hist_gam.append(g_t)
            if t % 10 == 0: self.hist_weights.append((t, w_t.copy(), gb_fit))
            if self.verbose and (t % 15 == 0 or t == 1 or t == self.n_iter):
                phase = "EXPLORE" if p < 0.5 else "EXPLOIT"
                print(f"  [{phase}] iter={t:4d}/{self.n_iter} | best={gb_fit:.4f} | mean={np.mean(snap):.4f} | C={C_t:7.3f} | γ={g_t:.6f} | α={α:.3f}")
        self.best_weights_ = self._decode(gb_pos)[0]
        self.best_C_ = self._decode(gb_pos)[1]
        self.best_gamma_ = self._decode(gb_pos)[2]
        self.best_fitness_ = gb_fit
        return self

print('DeepDOA class defined')

DeepDOA class defined


## Section 3 — Run Optimisation

In [6]:

# 3. RUN OPTIMISATION
print("  Running Deep DOA  (n_droplets=25, n_iter=2000, cv=5-fold) …\n")
t0 = time.time()

doa = DeepDOA(
    n_droplets=25, n_iter=2000,
    w_min=0.001, w_max=5.0,
    logC_min=-1.0, logC_max=3.0,
    logg_min=-3.0, logg_max=1.0,
    alpha_max=0.95, alpha_min=0.40,
    bl_init=0.30, bl_end=0.62,
    bg_init=0.65, bg_end=0.22,
    levy_init=0.10, levy_min=0.003,
    cond_pct=0.20, cond_sigma=0.15,
    surge_every=15, surge_str=0.65,
    nb_init=4, nb_max=12,
    elite_k=3, cv_folds=5,
    verbose=True,
)
doa.optimize(X_raw, y)
elapsed = time.time() - t0

best_w   = doa.best_weights_
best_C   = doa.best_C_
best_gam = doa.best_gamma_
best_cv  = doa.best_fitness_

print(f"\n  ✔  Done in {elapsed/60:.1f} min")
print(f"  Best CV Accuracy : {best_cv:.4f}  ({best_cv*100:.2f}%)")
print(f"  Best C           : {best_C:.4f}")
print(f"  Best gamma       : {best_gam:.6f}")

  Running Deep DOA  (n_droplets=25, n_iter=2000, cv=5-fold) …

  Init    | best=0.8291 | C=0.7672 | γ=0.157033
  [EXPLORE] iter=   1/2000 | best=0.8320 | mean=0.8119 | C=  1.302 | γ=0.178405 | α=0.950
  [EXPLORE] iter=  15/2000 | best=0.8320 | mean=0.8217 | C=  1.302 | γ=0.178405 | α=0.950
  [EXPLORE] iter=  30/2000 | best=0.8330 | mean=0.8250 | C=  1.721 | γ=0.121274 | α=0.950
  [EXPLORE] iter=  45/2000 | best=0.8330 | mean=0.8212 | C=  1.142 | γ=0.197734 | α=0.949
  [EXPLORE] iter=  60/2000 | best=0.8330 | mean=0.8193 | C=  1.142 | γ=0.197734 | α=0.949
  [EXPLORE] iter=  75/2000 | best=0.8330 | mean=0.8217 | C=  1.142 | γ=0.197734 | α=0.948
  [EXPLORE] iter=  90/2000 | best=0.8330 | mean=0.8210 | C=  1.142 | γ=0.197734 | α=0.947
  [EXPLORE] iter= 105/2000 | best=0.8330 | mean=0.8168 | C=  1.142 | γ=0.197734 | α=0.946
  [EXPLORE] iter= 120/2000 | best=0.8330 | mean=0.8249 | C=  1.142 | γ=0.197734 | α=0.945
  [EXPLORE] iter= 135/2000 | best=0.8330 | mean=0.8188 | C=  1.142 | γ=0.197734

## Section 4 — Exploratory Data Analysis (EDA)

In [7]:

# 4. EXPLORATORY DATA ANALYSIS VISUALISATIONS

print("\n  Generating EDA visualisations …")

from scipy import stats

fig_eda = plt.figure(figsize=(20, 14))
gs_eda  = gridspec.GridSpec(3, 3, figure=fig_eda, hspace=0.40, wspace=0.35)
colors_palette = ['#FF6B6B', '#4ECDC4']

ax_4_1 = fig_eda.add_subplot(gs_eda[0, 0])
target_counts = pd.Series(y).value_counts()
bars_target = ax_4_1.bar(['No Disease', 'Disease'],
                         [target_counts[0], target_counts[1]],
                         color=colors_palette, edgecolor='white', linewidth=1.5, width=0.5)
ax_4_1.set_title('4.1 Target Distribution', fontsize=12, fontweight='bold')
ax_4_1.set_ylabel('Count', fontsize=10)
for bar, count in zip(bars_target, [target_counts[0], target_counts[1]]):
    height = bar.get_height()
    ax_4_1.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(count)}\n({count/len(y)*100:.1f}%)',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

for idx, feat in enumerate(FEATURES[:3]):
    ax = fig_eda.add_subplot(gs_eda[1, idx % 3])
    feat_no_disease = X_raw[y == 0, FEATURES.index(feat)]
    feat_disease = X_raw[y == 1, FEATURES.index(feat)]
    ax.hist(feat_no_disease, bins=15, alpha=0.6, label='No Disease', color=colors_palette[0], edgecolor='white')
    ax.hist(feat_disease, bins=15, alpha=0.6, label='Disease', color=colors_palette[1], edgecolor='white')
    ax.set_title(f'4.2 {feat}', fontsize=10, fontweight='bold')
    ax.set_xlabel(feat, fontsize=9); ax.set_ylabel('Frequency', fontsize=9)
    ax.legend(fontsize=8); ax.grid(alpha=0.3, linestyle='--')

ax_4_4 = fig_eda.add_subplot(gs_eda[2, 0])
if 'sex' in FEATURES:
    sex_disease = data[data['target'] == 1]['sex'].value_counts()
    sex_no_disease = data[data['target'] == 0]['sex'].value_counts()
    x_pos = np.arange(2); width = 0.35
    ax_4_4.bar(x_pos - width/2, [sex_no_disease.get(0, 0), sex_no_disease.get(1, 0)],
               width, label='No Disease', color=colors_palette[0], edgecolor='white')
    ax_4_4.bar(x_pos + width/2, [sex_disease.get(0, 0), sex_disease.get(1, 0)],
               width, label='Disease', color=colors_palette[1], edgecolor='white')
    ax_4_4.set_title('4.4 Sex Distribution', fontsize=10, fontweight='bold')
    ax_4_4.set_ylabel('Count', fontsize=9); ax_4_4.set_xticks(x_pos)
    ax_4_4.set_xticklabels(['Female (0)', 'Male (1)'], fontsize=9)
    ax_4_4.legend(fontsize=8); ax_4_4.grid(axis='y', alpha=0.3, linestyle='--')

ax_4_4_2 = fig_eda.add_subplot(gs_eda[2, 1])
if 'cp' in FEATURES:
    cp_disease = data[data['target'] == 1]['cp'].value_counts().sort_index()
    cp_no_disease = data[data['target'] == 0]['cp'].value_counts().sort_index()
    n_cp = max(len(cp_disease), len(cp_no_disease))
    x_pos = np.arange(n_cp); width = 0.35
    ax_4_4_2.bar(x_pos - width/2, [cp_no_disease.get(i, 0) for i in range(n_cp)],
                 width, label='No Disease', color=colors_palette[0], edgecolor='white')
    ax_4_4_2.bar(x_pos + width/2, [cp_disease.get(i, 0) for i in range(n_cp)],
                 width, label='Disease', color=colors_palette[1], edgecolor='white')
    ax_4_4_2.set_title('4.4 Chest Pain Type (cp)', fontsize=10, fontweight='bold')
    ax_4_4_2.set_ylabel('Count', fontsize=9); ax_4_4_2.set_xticks(x_pos)
    ax_4_4_2.set_xticklabels([f'Type {i}' for i in range(n_cp)], fontsize=9)
    ax_4_4_2.legend(fontsize=8); ax_4_4_2.grid(axis='y', alpha=0.3, linestyle='--')

ax_4_4_3 = fig_eda.add_subplot(gs_eda[2, 2])
if 'thal' in FEATURES:
    thal_disease = data[data['target'] == 1]['thal'].value_counts().sort_index()
    thal_no_disease = data[data['target'] == 0]['thal'].value_counts().sort_index()
    n_thal = max(len(thal_disease), len(thal_no_disease))
    x_pos = np.arange(n_thal); width = 0.35
    ax_4_4_3.bar(x_pos - width/2, [thal_no_disease.get(i, 0) for i in range(n_thal)],
                 width, label='No Disease', color=colors_palette[0], edgecolor='white')
    ax_4_4_3.bar(x_pos + width/2, [thal_disease.get(i, 0) for i in range(n_thal)],
                 width, label='Disease', color=colors_palette[1], edgecolor='white')
    ax_4_4_3.set_title('4.4 Thalassemia (thal)', fontsize=10, fontweight='bold')
    ax_4_4_3.set_ylabel('Count', fontsize=9); ax_4_4_3.set_xticks(x_pos)
    ax_4_4_3.set_xticklabels([f'Type {i}' for i in range(n_thal)], fontsize=9)
    ax_4_4_3.legend(fontsize=8); ax_4_4_3.grid(axis='y', alpha=0.3, linestyle='--')

fig_eda.suptitle('EDA Visualisations — UCI Heart Disease Dataset\nTarget Distribution, Numerical Features KDE, & Categorical Features',
                fontsize=13, fontweight='bold', y=0.995)
eda_out = os.path.expanduser('~/Downloads/heart_eda_visualisations.png')
plt.savefig(eda_out, dpi=150, bbox_inches='tight', facecolor='white')
print(f"  [EDA PLOT] Saved → {eda_out}")
plt.show()


  Generating EDA visualisations …
  [EDA PLOT] Saved → /Users/minhquan/Downloads/heart_eda_visualisations.png


## Section 4.6 — Pairplot: Numerical Features

In [8]:

# 4.6 PAIRPLOT — NUMERICAL FEATURES

print("\n  Generating Pairplot visualization …")
import seaborn as sns

key_features_for_pair = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
key_features_idx = [FEATURES.index(f) for f in key_features_for_pair if f in FEATURES]
key_features_names = [f for f in key_features_for_pair if f in FEATURES]

pair_data = pd.DataFrame(X_raw[:, key_features_idx], columns=key_features_names)
pair_data['target'] = y
pair_data['target_label'] = pair_data['target'].map({0: 'No Disease', 1: 'Disease'})

fig_pair = sns.pairplot(pair_data, hue='target_label',
                        palette={'No Disease': '#FF6B6B', 'Disease': '#4ECDC4'},
                        diag_kind='kde', plot_kws={'alpha': 0.6, 's': 30},
                        corner=False, height=2.0)
fig_pair.fig.suptitle('4.6 Pairplot — Numerical Features\nRelationships between Age, Blood Pressure, Cholesterol, Heart Rate & ST Depression',
                      fontsize=12, fontweight='bold', y=0.995)
pair_out = os.path.expanduser('~/Downloads/heart_pairplot.png')
plt.savefig(pair_out, dpi=150, bbox_inches='tight', facecolor='white')
print(f"  [PAIRPLOT] Saved → {pair_out}")
plt.show()
plt.close(fig_pair.fig)


  Generating Pairplot visualization …
  [PAIRPLOT] Saved → /Users/minhquan/Downloads/heart_pairplot.png


## Section 4.7 — Regression Plots: Age vs Key Features

In [9]:
# 4.7 REGRESSION PLOTS — AGE VS KEY FEATURES
print("  Generating Regression Plots …")

regression_features = ['trestbps', 'chol', 'thalach', 'oldpeak']
reg_features_idx = [FEATURES.index(f) for f in regression_features if f in FEATURES]
reg_features_names = [f for f in regression_features if f in FEATURES]

fig_reg = plt.figure(figsize=(16, 10))
gs_reg = gridspec.GridSpec(2, 2, figure=fig_reg, hspace=0.35, wspace=0.30)
age_idx = FEATURES.index('age')

for plot_idx, (feat_name, feat_idx) in enumerate(zip(reg_features_names, reg_features_idx)):
    ax_reg = fig_reg.add_subplot(gs_reg[plot_idx // 2, plot_idx % 2])
    for target_val, label, color in [(0, 'No Disease', '#FF6B6B'), (1, 'Disease', '#4ECDC4')]:
        mask = y == target_val
        ax_reg.scatter(X_raw[mask, age_idx], X_raw[mask, feat_idx],
                      alpha=0.5, s=25, color=color, label=label, edgecolors='none')
    z_nd = np.polyfit(X_raw[y == 0, age_idx], X_raw[y == 0, feat_idx], 1)
    p_nd = np.poly1d(z_nd)
    age_range = np.linspace(X_raw[:, age_idx].min(), X_raw[:, age_idx].max(), 100)
    ax_reg.plot(age_range, p_nd(age_range), '--', color='#FF6B6B', lw=2.5, alpha=0.8, label='Trend: No Disease')
    z_d = np.polyfit(X_raw[y == 1, age_idx], X_raw[y == 1, feat_idx], 1)
    p_d = np.poly1d(z_d)
    ax_reg.plot(age_range, p_d(age_range), '--', color='#4ECDC4', lw=2.5, alpha=0.8, label='Trend: Disease')
    ax_reg.set_xlabel('Age (years)', fontsize=10, fontweight='bold')
    ax_reg.set_ylabel(feat_name.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    ax_reg.set_title(f'Age vs {feat_name.upper()}', fontsize=11, fontweight='bold')
    ax_reg.legend(fontsize=9, loc='best'); ax_reg.grid(alpha=0.3, linestyle='--')

fig_reg.suptitle('4.7 Regression Plots — Age vs Key Features\nLinear Regression by Disease Status',
                fontsize=12, fontweight='bold', y=0.995)
reg_out = os.path.expanduser('~/Downloads/heart_regression_plots.png')
plt.savefig(reg_out, dpi=150, bbox_inches='tight', facecolor='white')
print(f"  [REGRESSION PLOTS] Saved → {reg_out}\n")
plt.show()

  Generating Regression Plots …
  [REGRESSION PLOTS] Saved → /Users/minhquan/Downloads/heart_regression_plots.png



## Section 4 (Final) — Hold-out Evaluation

In [10]:
# 4. FINAL HOLD-OUT EVALUATION
X_w = X_raw * best_w
X_tr, X_te, y_tr, y_te = train_test_split(X_w, y, test_size=0.20, random_state=42, stratify=y)

sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s = sc.transform(X_te)

svm = SVC(C=best_C, kernel='rbf', gamma=best_gam, probability=True, random_state=42)
svm.fit(X_tr_s, y_tr)
yp    = svm.predict(X_te_s)
yprob = svm.predict_proba(X_te_s)[:, 1]

acc = accuracy_score(y_te, yp)
auc = roc_auc_score(y_te, yprob)
f1  = f1_score(y_te, yp)
mcc = matthews_corrcoef(y_te, yp)
ap  = average_precision_score(y_te, yprob)

print("\n" + "═"*70)
print("  FINAL RESULTS (20% hold-out test set)")
print("═"*70)
print(f"  Accuracy  : {acc*100:.2f}%")
print(f"  AUC-ROC   : {auc:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print(f"  MCC       : {mcc:.4f}")
print(f"  Avg Prec  : {ap:.4f}")
print()
print(classification_report(y_te, yp, target_names=['No Disease','Disease']))


══════════════════════════════════════════════════════════════════════
  FINAL RESULTS (20% hold-out test set)
══════════════════════════════════════════════════════════════════════
  Accuracy  : 81.37%
  AUC-ROC   : 0.8854
  F1-Score  : 0.8061
  MCC       : 0.6294
  Avg Prec  : 0.8871

              precision    recall  f1-score   support

  No Disease       0.79      0.85      0.82       102
     Disease       0.84      0.77      0.81       102

    accuracy                           0.81       204
   macro avg       0.82      0.81      0.81       204
weighted avg       0.82      0.81      0.81       204



## Section 4B — SVM Analysis (Support Vectors, Decision Function, Kernel Comparison)

In [11]:
# 4B. PHAN TICH SUPPORT VECTOR MACHINE
print("\n  Generating SVM Analysis visualisations …")

from sklearn.decomposition import PCA

# PCA 2D for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_tr_s)
X_te_pca = pca.transform(X_te_s)

# Train SVM on PCA data for visualization
svm_pca = SVC(C=best_C, kernel='rbf', gamma=best_gam, probability=True, random_state=42)
svm_pca.fit(X_pca, y_tr)

fig_svm = plt.figure(figsize=(20, 6))
gs_svm = gridspec.GridSpec(1, 3, figure=fig_svm, hspace=0.3, wspace=0.35)

# ── Panel 1: Support Vectors visualization ────────────────────────────────
ax_sv = fig_svm.add_subplot(gs_svm[0, 0])

x_min, x_max = X_pca[:, 0].min() - 0.5, X_pca[:, 0].max() + 0.5
y_min, y_max = X_pca[:, 1].min() - 0.5, X_pca[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
Z = svm_pca.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

ax_sv.contourf(xx, yy, Z, alpha=0.2, cmap='RdYlGn')
colors_cls = ['#e74c3c', '#2ecc71']
for cls, color, label in [(0, colors_cls[0], 'No Disease'), (1, colors_cls[1], 'Disease')]:
    mask = y_tr == cls
    ax_sv.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, s=20, alpha=0.6, label=label, edgecolors='none')

sv_idx = svm_pca.support_
ax_sv.scatter(X_pca[sv_idx, 0], X_pca[sv_idx, 1], s=80, facecolors='none',
              edgecolors='blue', linewidths=1.5, label=f'Support Vectors: {len(sv_idx)}/{len(y_tr)} ({len(sv_idx)/len(y_tr)*100:.1f}%)')

ax_sv.set_xlabel('PCA Component 1', fontsize=10)
ax_sv.set_ylabel('PCA Component 2', fontsize=10)
ax_sv.set_title(f'SVM (kernel=rbf, C={best_C:.1f}) - Support Vectors', fontsize=11, fontweight='bold')
ax_sv.legend(fontsize=8, loc='upper left')

# ── Panel 2: Decision Function & Margins ─────────────────────────────────
ax_df = fig_svm.add_subplot(gs_svm[0, 1])

Z_df = svm_pca.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z_df = Z_df.reshape(xx.shape)

cf = ax_df.contourf(xx, yy, Z_df, levels=20, cmap='RdBu_r', alpha=0.8)
plt.colorbar(cf, ax=ax_df)
ax_df.contour(xx, yy, Z_df, levels=[0], colors='black', linewidths=2)
ax_df.contour(xx, yy, Z_df, levels=[-1, 1], colors='black', linewidths=1, linestyles='--')

for cls, color in [(0, colors_cls[0]), (1, colors_cls[1])]:
    mask = y_tr == cls
    ax_df.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, s=15, alpha=0.5, edgecolors='none')
ax_df.scatter(X_pca[sv_idx, 0], X_pca[sv_idx, 1], s=60, facecolors='none',
              edgecolors='blue', linewidths=1.5)

ax_df.set_xlabel('PCA Component 1', fontsize=10)
ax_df.set_ylabel('PCA Component 2', fontsize=10)
ax_df.set_title('SVM - Decision Function & Margins', fontsize=11, fontweight='bold')

# ── Panel 3: Kernel Comparison (C effect) ────────────────────────────────
ax_kc = fig_svm.add_subplot(gs_svm[0, 2])

C_values = [0.01, 0.1, 1, 10, 100]
kernels = ['linear', 'rbf', 'poly']
kernel_colors = {'linear': '#e74c3c', 'rbf': '#3498db', 'poly': '#2ecc71'}

kf_comp = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
kernel_results = {k: [] for k in kernels}
for C_val in C_values:
    for kern in kernels:
        svm_k = SVC(C=C_val, kernel=kern, random_state=42)
        score = cross_val_score(svm_k, X_pca, y_tr, cv=kf_comp, scoring='accuracy').mean()
        kernel_results[kern].append(score)

x_pos = np.arange(len(C_values))
width = 0.25
for i, (kern, color) in enumerate(kernel_colors.items()):
    offset = (i - 1) * width
    ax_kc.bar(x_pos + offset, kernel_results[kern], width, label=kern.upper(),
              color=color, alpha=0.8, edgecolor='white')

ax_kc.set_xlabel('Gia tri C (regularization)', fontsize=10)
ax_kc.set_ylabel('Test Accuracy', fontsize=10)
ax_kc.set_title('SVM - Anh huong cua Kernel va C', fontsize=11, fontweight='bold')
ax_kc.set_xticks(x_pos)
ax_kc.set_xticklabels([str(c) for c in C_values], fontsize=9)
ax_kc.legend(fontsize=9)
ax_kc.set_ylim(0, 1.05)
ax_kc.grid(axis='y', alpha=0.3, linestyle='--')

fig_svm.suptitle('PHAN TICH SUPPORT VECTOR MACHINE\nSupport Vectors | Decision Function & Margins | Kernel vs C Comparison',
                fontsize=13, fontweight='bold', y=1.02)

svm_out = os.path.expanduser('~/Downloads/heart_svm_analysis.png')
plt.savefig(svm_out, dpi=150, bbox_inches='tight', facecolor='white')
print(f"  [SVM PLOT] Saved → {svm_out}")
plt.show()

print(f"\n  Support Vectors: {len(sv_idx)} / {len(y_tr)} ({len(sv_idx)/len(y_tr)*100:.1f}%)")
print(f"  Best C = {best_C:.4f} | Best gamma = {best_gam:.6f}")


  Generating SVM Analysis visualisations …
  [SVM PLOT] Saved → /Users/minhquan/Downloads/heart_svm_analysis.png

  Support Vectors: 373 / 814 (45.8%)
  Best C = 0.9406 | Best gamma = 0.131344


## Section 5 — Full Visualisation (12 panels)

In [12]:
# 5. FULL VISUALISATION  (12 panels)
print("  Rendering visualisations …")

importance      = best_w * np.std(X_raw, axis=0)
importance_norm = importance / importance.sum()
si              = np.argsort(importance_norm)

BLUE   = '#1565C0'; ORANGE = '#E65100'; GREEN  = '#2E7D32'
PURPLE = '#6A1B9A'; TEAL   = '#00695C'; RED    = '#C62828'
LBLUE  = '#42A5F5'; GOLD   = '#F57F17'

fig = plt.figure(figsize=(22, 28))
gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.50, wspace=0.38)
iters = np.arange(len(doa.hist_best))

ax0 = fig.add_subplot(gs[0, :2])
mean_arr = np.array(doa.hist_mean); std_arr = np.array(doa.hist_std)
ax0.fill_between(iters, np.clip(mean_arr - std_arr, 0, 1), np.clip(mean_arr + std_arr, 0, 1),
                 alpha=0.12, color=ORANGE, label='Mean ± 1 std')
ax0.fill_between(iters, mean_arr, doa.hist_best, alpha=0.10, color=BLUE)
ax0.plot(iters, doa.hist_best, color=BLUE, lw=2.5, label=f'Global Best → {best_cv*100:.2f}%')
ax0.plot(iters, mean_arr, '--', color=ORANGE, lw=1.8, alpha=0.9, label='Population Mean')
ax0.axhline(0.75, color=GREEN, lw=1.4, ls=':', label='Target 75%')
ax0.axvline(len(iters)//2, color='grey', lw=1.2, ls='--', alpha=0.5, label='Phase switch (Explore→Exploit)')
ax0.set_title('DOA Convergence — Global Best & Population Mean (5-fold CV Accuracy)', fontsize=13, fontweight='bold')
ax0.set_xlabel('Iteration'); ax0.set_ylabel('CV Accuracy')
ax0.legend(fontsize=9, loc='lower right')
ax0.set_ylim(max(0.60, min(mean_arr) - 0.03), 1.01)

ax1 = fig.add_subplot(gs[0, 2])
p_iters = np.arange(1, len(doa.alpha_hist)+1)
ax1.plot(p_iters, doa.alpha_hist, color=BLUE, lw=2, label='α (evaporation)')
ax1.plot(p_iters, doa.bl_hist, color=GREEN, lw=2, label='β_local')
ax1.plot(p_iters, doa.bg_hist, color=RED, lw=2, label='β_global')
ax1.plot(p_iters, [l*5 for l in doa.levy_hist], color=GOLD, lw=1.5, ls='--', label='Lévy scale ×5')
ax1.set_title('Adaptive Parameter\nSchedules', fontsize=11, fontweight='bold')
ax1.set_xlabel('Iteration'); ax1.set_ylabel('Value')
ax1.legend(fontsize=8)

ax2 = fig.add_subplot(gs[1, 0])
ax2.semilogy(iters, doa.hist_C, color=PURPLE, lw=2, alpha=0.85)
ax2.axhline(best_C, color=RED, lw=1.5, ls='--', label=f'Best C = {best_C:.4f}')
ax2.set_title('SVM Regularization C\nEvolution', fontsize=11, fontweight='bold')
ax2.set_xlabel('Iteration'); ax2.set_ylabel('C (log scale)')
ax2.legend(fontsize=9)

ax3 = fig.add_subplot(gs[1, 1])
ax3.semilogy(iters, doa.hist_gam, color=TEAL, lw=2, alpha=0.85)
ax3.axhline(best_gam, color=RED, lw=1.5, ls='--', label=f'Best γ = {best_gam:.6f}')
ax3.set_title('SVM Kernel γ (gamma)\nEvolution', fontsize=11, fontweight='bold')
ax3.set_xlabel('Iteration'); ax3.set_ylabel('γ (log scale)')
ax3.legend(fontsize=9)

ax4 = fig.add_subplot(gs[1, 2])
sc_plot = ax4.scatter(np.log10(doa.hist_C), np.log10(doa.hist_gam),
                      c=doa.hist_best, cmap='plasma', s=20, alpha=0.55, edgecolors='none')
ax4.scatter(np.log10(best_C), np.log10(best_gam), color='red', s=220, marker='*', zorder=10,
            label=f'Best  C={best_C:.2f}  γ={best_gam:.5f}')
cbar = plt.colorbar(sc_plot, ax=ax4, pad=0.02); cbar.set_label('CV Accuracy', fontsize=9)
ax4.set_title('Hyperparameter Search\nlog₁₀(C) vs log₁₀(γ)', fontsize=11, fontweight='bold')
ax4.set_xlabel('log₁₀(C)'); ax4.set_ylabel('log₁₀(γ)')
ax4.legend(fontsize=8)

ax5 = fig.add_subplot(gs[2, 0])
col_w = [GREEN if w >= 1.0 else RED for w in best_w]
bars = ax5.barh(FEATURES, best_w, color=col_w, edgecolor='white', height=0.65, linewidth=0.7)
ax5.axvline(1.0, color='grey', lw=1.4, ls='--', alpha=0.7, label='Uniform = 1')
ax5.set_title('Optimised Feature Weights\n(green ≥ 1, red < 1)', fontsize=11, fontweight='bold')
ax5.set_xlabel('Weight'); ax5.legend(fontsize=9)
for bar, w in zip(bars, best_w):
    ax5.text(w + 0.04, bar.get_y() + bar.get_height()/2, f'{w:.3f}', va='center', fontsize=8.5)

ax6 = fig.add_subplot(gs[2, 1])
grad = plt.cm.YlOrRd(np.linspace(0.3, 0.95, n_feat))
ax6.barh(np.array(FEATURES)[si], importance_norm[si], color=grad, edgecolor='white', linewidth=0.6)
ax6.set_title('Feature Importance\n(weight × std, normalised)', fontsize=11, fontweight='bold')
ax6.set_xlabel('Relative Importance', fontsize=10)
for j, v in enumerate(importance_norm[si]):
    ax6.text(v + 0.003, j, f'{v:.3f}', va='center', fontsize=8.5)

ax7 = fig.add_subplot(gs[2, 2], polar=True)
angles = np.linspace(0, 2*np.pi, n_feat, endpoint=False).tolist()
vals = best_w.tolist()
angles_c = angles + angles[:1]; vals_c = vals + vals[:1]
ax7.plot(angles_c, vals_c, color=BLUE, lw=2.2)
ax7.fill(angles_c, vals_c, alpha=0.20, color=BLUE)
ax7.set_xticks(angles)
ax7.set_xticklabels(FEATURES, fontsize=8.5)
ax7.set_title('Feature Weight\nRadar Chart', fontsize=11, fontweight='bold', pad=18)

ax8 = fig.add_subplot(gs[3, 0])
cm_val = confusion_matrix(y_te, yp)
ConfusionMatrixDisplay(cm_val, display_labels=['No Disease','Disease']).plot(ax=ax8, colorbar=False, cmap='Blues')
ax8.set_title(f'Confusion Matrix\nAcc={acc*100:.2f}%   F1={f1:.3f}   MCC={mcc:.3f}', fontsize=11, fontweight='bold')

ax9 = fig.add_subplot(gs[3, 1])
fpr, tpr, thr = roc_curve(y_te, yprob)
ax9.fill_between(fpr, tpr, alpha=0.12, color=BLUE)
ax9.plot(fpr, tpr, color=BLUE, lw=2.5, label=f'DOA-SVM  AUC = {auc:.4f}')
ax9.plot([0,1],[0,1],'k--', lw=1, alpha=0.4, label='Random')
opt = np.argmax(tpr - fpr)
ax9.scatter(fpr[opt], tpr[opt], color=RED, s=100, zorder=6, label=f'Best thresh ≈ {thr[opt]:.2f}')
ax9.set_title('ROC Curve', fontsize=11, fontweight='bold')
ax9.set_xlabel('False Positive Rate'); ax9.set_ylabel('True Positive Rate')
ax9.legend(fontsize=9)

ax10 = fig.add_subplot(gs[3, 2])
prec, rec, _ = precision_recall_curve(y_te, yprob)
ax10.fill_between(rec, prec, alpha=0.12, color=TEAL)
ax10.plot(rec, prec, color=TEAL, lw=2.5, label=f'AP = {ap:.4f}')
ax10.axhline(y_te.mean(), color='grey', ls='--', lw=1.2, label=f'Baseline = {y_te.mean():.2f}')
ax10.set_title('Precision-Recall Curve', fontsize=11, fontweight='bold')
ax10.set_xlabel('Recall'); ax10.set_ylabel('Precision')
ax10.legend(fontsize=9)

fig.suptitle(
    'Deep DOA-SVM │ UCI Heart Disease (Cleveland + Hungarian + VA + Switzerland)\n'
    f'Accuracy={acc*100:.2f}%   AUC={auc:.4f}   F1={f1:.4f}   MCC={mcc:.4f}   AP={ap:.4f}\n'
    f'Optimised: C={best_C:.4f}  γ={best_gam:.6f}  │  n_droplets=25  n_iter=120  cv=5-fold  runtime={elapsed/60:.1f}min',
    fontsize=13, fontweight='bold', y=0.999
)

out = os.path.expanduser('~/Downloads/heart_drizzle_svm_results.png')
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
print(f"  [PLOT] Saved → {out}")
plt.show()

  Rendering visualisations …
  [PLOT] Saved → /Users/minhquan/Downloads/heart_drizzle_svm_results.png


## Section 6 — Final Summary

In [13]:
# 6. FINAL SUMMARY
print("\n" + "═"*70)
print("  FINAL SUMMARY")
print("═"*70)
print(f"  CV Accuracy   : {best_cv*100:.2f}%")
print(f"  Test Accuracy : {acc*100:.2f}%")
print(f"  AUC-ROC       : {auc:.4f}")
print(f"  F1-Score      : {f1:.4f}")
print(f"  MCC           : {mcc:.4f}")
print(f"  Avg Precision : {ap:.4f}")
print(f"  Best C        : {best_C:.4f}")
print(f"  Best gamma    : {best_gam:.6f}")
print(f"  Runtime       : {elapsed/60:.1f} min")
print()
print("  Feature weights (sorted by importance):")
sorted_feat = sorted(zip(FEATURES, best_w, importance_norm), key=lambda x: -x[2])
for feat, w, imp in sorted_feat:
    bar = '█' * int(imp * 70)
    print(f"    {feat:<12}  w={w:.4f}  imp={imp:.4f}  {bar}")
print("\n  ✔  All done.")


══════════════════════════════════════════════════════════════════════
  FINAL SUMMARY
══════════════════════════════════════════════════════════════════════
  CV Accuracy   : 83.40%
  Test Accuracy : 81.37%
  AUC-ROC       : 0.8854
  F1-Score      : 0.8061
  MCC           : 0.6294
  Avg Precision : 0.8871
  Best C        : 0.9406
  Best gamma    : 0.131344
  Runtime       : 35.7 min

  Feature weights (sorted by importance):
    chol          w=1.3762  imp=0.4819  █████████████████████████████████
    trestbps      w=3.6207  imp=0.2143  ██████████████
    thalach       w=2.4058  imp=0.1975  █████████████
    age           w=1.7194  imp=0.0529  ███
    oldpeak       w=4.9375  imp=0.0166  █
    cp            w=3.8691  imp=0.0117  
    restecg       w=3.3764  imp=0.0088  
    slope         w=3.7228  imp=0.0062  
    fbs           w=3.5938  imp=0.0041  
    exang         w=2.1291  imp=0.0033  
    sex           w=2.0425  imp=0.0028  

  ✔  All done.


## Section 7 — Flask Web App + Ngrok

In [14]:
# 7. FLASK WEB APP + NGROK
prediction_history = []
CSV_FILE = 'heart_disease_predictions.csv'

CSV_HEADER = [
    'Thoi_gian', 'Ten_benh_nhan', 'Tuoi', 'Gioi_tinh', 'Chieu_cao_cm', 'Can_nang_kg', 'BMI',
    'Loai_dau_nguc', 'Huyet_ap_mmHg', 'Cholesterol_mgdl', 'Duong_huyet_cao',
    'Ket_qua_ECG', 'Nhip_tim_toi_da_bpm', 'Dau_nguc_van_dong',
    'ST_Depression', 'Do_doc_ST', 'So_mach_mau_chinh', 'Thalassemia',
    'Ket_qua_du_doan', 'Xac_suat_benh_tim_%', 'Xac_suat_khoe_manh_%',
    'Do_tin_cay_%', 'Model', 'Phan_loai'
]

# Create CSV with header if not exists
if not os.path.exists(CSV_FILE):
    with open(CSV_FILE, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(CSV_HEADER)


def predict_heart_disease(features_list, patient_name, patient_data):
    """Predict using trained DOA-SVM model"""
    # Scale using saved scaler
    feat_arr = np.array(features_list).reshape(1, -1)
    feat_weighted = feat_arr * best_w
    feat_scaled = sc.transform(feat_weighted)
    
    pred = svm.predict(feat_scaled)[0]
    prob = svm.predict_proba(feat_scaled)[0]
    
    disease_prob = float(prob[1]) * 100
    healthy_prob = float(prob[0]) * 100
    confidence = max(disease_prob, healthy_prob)
    
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    result = {
        'timestamp': timestamp,
        'patient_name': patient_name,
        'prediction': int(pred),
        'disease_probability': disease_prob,
        'healthy_probability': healthy_prob,
        'confidence': confidence,
        'model_name': f'DOA-SVM (C={best_C:.3f}, γ={best_gam:.5f})'
    }
    
    prediction_history.append(result)
    _save_to_csv(result, patient_data)
    return result


def _save_to_csv(result, patient_data):
    try:
        height_m = patient_data.get('height', 170) / 100
        weight = patient_data.get('weight', 70)
        bmi = round(weight / (height_m * height_m), 2)
        sex_map = {1: 'Nam', 0: 'Nữ'}
        cp_map = {1: 'Typical Angina', 2: 'Atypical Angina', 3: 'Non-anginal', 4: 'Asymptomatic'}
        ecg_map = {0: 'Normal', 1: 'ST-T Abnormality', 2: 'LV Hypertrophy'}
        slope_map = {1: 'Upsloping', 2: 'Flat', 3: 'Downsloping'}
        thal_map = {3: 'Normal', 6: 'Fixed Defect', 7: 'Reversible Defect'}
        row = [
            result['timestamp'], result['patient_name'],
            patient_data.get('age', 'N/A'), sex_map.get(patient_data.get('sex', 1), 'N/A'),
            patient_data.get('height', 'N/A'), patient_data.get('weight', 'N/A'), bmi,
            cp_map.get(patient_data.get('cp', 1), 'N/A'),
            patient_data.get('trestbps', 'N/A'), patient_data.get('chol', 'N/A'),
            'Có' if patient_data.get('fbs', 0) == 1 else 'Không',
            ecg_map.get(patient_data.get('restecg', 0), 'N/A'),
            patient_data.get('thalach', 'N/A'),
            'Có' if patient_data.get('exang', 0) == 1 else 'Không',
            patient_data.get('oldpeak', 'N/A'),
            slope_map.get(patient_data.get('slope', 1), 'N/A'),
            patient_data.get('ca', 'N/A'), thal_map.get(patient_data.get('thal', 3), 'N/A'),
            'Nguy cơ bệnh tim' if result['prediction'] == 1 else 'Khỏe mạnh',
            round(result['disease_probability'], 2), round(result['healthy_probability'], 2),
            round(result['confidence'], 2), result['model_name'],
            'Có bệnh' if result['prediction'] == 1 else 'Không bệnh'
        ]
        with open(CSV_FILE, 'a', newline='', encoding='utf-8-sig') as f:
            writer = csv.writer(f)
            writer.writerow(row)
        print(f"Đã lưu CSV: {result['patient_name']}")
    except Exception as e:
        print(f"Lỗi CSV: {e}")


print('✅ Prediction functions defined')

✅ Prediction functions defined


In [15]:

# 7B. HTML TEMPLATE

HTML_TEMPLATE = '''
<!DOCTYPE html>
<html lang="vi">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>💓 Hệ thống Dự đoán Bệnh Tim</title>
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body {
            font-family: \'Segoe UI\', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            padding: 20px;
        }
        .header { text-align: center; color: white; margin-bottom: 30px; }
        .header h1 { font-size: 2.8em; margin-bottom: 10px; text-shadow: 2px 2px 4px rgba(0,0,0,0.3); }
        .model-badge {
            background: rgba(255,255,255,0.2); padding: 10px 25px;
            border-radius: 25px; display: inline-block; margin-top: 15px;
            font-size: 1.1em; backdrop-filter: blur(10px);
        }
        .container {
            max-width: 1600px; margin: 0 auto;
            display: grid; grid-template-columns: 420px 1fr; gap: 25px;
        }
        .card {
            background: white; border-radius: 20px;
            box-shadow: 0 15px 50px rgba(0,0,0,0.3); padding: 30px;
        }
        .card h2 {
            color: #667eea; margin-bottom: 20px; font-size: 1.6em;
            border-bottom: 3px solid #667eea; padding-bottom: 10px;
        }
        .stats-grid { display: grid; grid-template-columns: repeat(2, 1fr); gap: 15px; margin-bottom: 20px; }
        .stat-box {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white; padding: 20px; border-radius: 12px; text-align: center;
            box-shadow: 0 4px 15px rgba(0,0,0,0.2);
        }
        .stat-value { font-size: 2.2em; font-weight: bold; }
        .stat-label { font-size: 0.95em; margin-top: 5px; }
        .form-group { margin-bottom: 15px; }
        .form-group label { display: block; margin-bottom: 6px; color: #333; font-weight: 600; font-size: 0.95em; }
        .form-group input, .form-group select {
            width: 100%; padding: 12px; border: 2px solid #e0e0e0;
            border-radius: 10px; font-size: 1em; transition: all 0.3s;
        }
        .form-group input:focus, .form-group select:focus {
            border-color: #667eea; outline: none;
            box-shadow: 0 0 0 3px rgba(102, 126, 234, 0.1);
        }
        .btn-group { display: flex; gap: 10px; margin-top: 20px; }
        button {
            flex: 1; padding: 15px; border: none; border-radius: 12px;
            font-size: 1.05em; font-weight: 600; cursor: pointer; transition: all 0.3s;
        }
        .btn-primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; }
        .btn-primary:hover { transform: translateY(-2px); box-shadow: 0 8px 20px rgba(102, 126, 234, 0.4); }
        .btn-secondary { background: #f0f0f0; color: #667eea; }
        .btn-secondary:hover { background: #e0e0e0; }
        .btn-success { background: linear-gradient(135deg, #2ecc71, #27ae60); color: white; }
        .loading { display: none; text-align: center; padding: 50px; }
        .loading.show { display: block; }
        .spinner {
            border: 5px solid #f3f3f3; border-top: 5px solid #667eea;
            border-radius: 50%; width: 60px; height: 60px;
            animation: spin 1s linear infinite; margin: 0 auto 20px;
        }
        @keyframes spin { 0% { transform: rotate(0deg); } 100% { transform: rotate(360deg); } }
        .report-container { display: none; }
        .report-container.show { display: block; }
        .body-section { display: grid; grid-template-columns: 380px 1fr; gap: 30px; margin: 30px 0; }
        .body-container {
            background: linear-gradient(135deg, #f8f9ff, #fff5f8);
            border-radius: 20px; padding: 25px; box-shadow: 0 10px 30px rgba(0,0,0,0.1);
        }
        .body-container h4 { text-align: center; color: #667eea; margin-bottom: 20px; font-size: 1.3em; }
        .human-body-svg { width: 100%; max-width: 320px; margin: 0 auto; display: block; }
        @keyframes heartbeat { 0%, 100% { transform: scale(1); } 50% { transform: scale(1.1); } }
        .heart-healthy { animation: heartbeat 1.5s ease-in-out infinite; }
        .heart-risk { animation: heartbeat 0.8s ease-in-out infinite; }
        .body-stats { background: white; border-radius: 12px; padding: 15px; margin-top: 20px; }
        .body-stats-grid { display: grid; grid-template-columns: repeat(2, 1fr); gap: 12px; font-size: 0.95em; }
        .body-stats-item { padding: 8px; background: #f8f9fa; border-radius: 8px; }
        .metrics-container { display: grid; grid-template-columns: repeat(2, 1fr); gap: 20px; }
        .metric-box {
            background: white; border-radius: 15px; padding: 25px;
            box-shadow: 0 5px 20px rgba(0,0,0,0.08); border-left: 6px solid;
            transition: transform 0.3s;
        }
        .metric-box:hover { transform: translateY(-5px); box-shadow: 0 10px 35px rgba(0,0,0,0.15); }
        .metric-box.normal { border-color: #2ecc71; background: linear-gradient(135deg, #ffffff, #f0fff4); }
        .metric-box.warning { border-color: #f39c12; background: linear-gradient(135deg, #ffffff, #fffaf0); }
        .metric-box.danger { border-color: #e74c3c; background: linear-gradient(135deg, #ffffff, #fff5f5); }
        .metric-icon { font-size: 2.5em; margin-bottom: 10px; }
        .metric-label { font-size: 1em; color: #666; font-weight: 600; margin-bottom: 10px; }
        .metric-value { font-size: 2.3em; font-weight: bold; color: #333; line-height: 1; }
        .metric-unit { font-size: 0.9em; color: #999; margin-left: 5px; }
        .metric-status {
            display: inline-block; padding: 6px 15px; border-radius: 15px;
            font-size: 0.85em; font-weight: 600; margin-top: 10px;
        }
        .status-normal { background: #d4edda; color: #155724; }
        .status-warning { background: #fff3cd; color: #856404; }
        .status-danger { background: #f8d7da; color: #721c24; }
        .recommendations-section {
            background: linear-gradient(135deg, #fff5f5, #fffaf0);
            border-radius: 20px; padding: 30px; margin: 30px 0; border: 3px solid #f39c12;
        }
        .recommendations-section h4 { color: #d35400; font-size: 1.6em; margin-bottom: 20px; text-align: center; }
        .recommendation-item {
            background: white; padding: 18px; margin-bottom: 12px;
            border-left: 5px solid #f39c12; border-radius: 12px;
            display: flex; align-items: center; transition: all 0.3s;
        }
        .recommendation-item:hover { transform: translateX(10px); box-shadow: 0 5px 20px rgba(243, 156, 18, 0.2); }
        .recommendation-icon { font-size: 2em; margin-right: 15px; min-width: 45px; text-align: center; }
        .recommendation-text { font-size: 1em; color: #333; line-height: 1.5; }
        .history-container { max-height: 400px; overflow-y: auto; }
        .history-item { background: white; padding: 15px; margin-bottom: 10px; border-radius: 12px; border-left: 5px solid; }
        .history-item.healthy { border-left-color: #2ecc71; }
        .history-item.risk { border-left-color: #e74c3c; }
        .history-header { display: flex; justify-content: space-between; margin-bottom: 8px; }
        .history-name { font-weight: bold; font-size: 1.1em; color: #333; }
        .history-time { color: #666; font-size: 0.9em; }
        .history-result { display: flex; justify-content: space-between; align-items: center; }
        .history-status { font-size: 1.1em; font-weight: bold; }
        .history-probabilities { font-size: 0.9em; color: #666; }
        @media (max-width: 1200px) {
            .container { grid-template-columns: 1fr; }
            .body-section { grid-template-columns: 1fr; }
            .metrics-container { grid-template-columns: 1fr; }
        }
    </style>
</head>
<body>
    <div class="header">
        <h1>💓 HỆ THỐNG DỰ ĐOÁN BỆNH TIM</h1>
        <p>Phân tích AI với Mô hình Cơ thể 2D + Báo cáo Chi tiết + Lưu CSV</p>
        <div class="model-badge">
            🏆 Model: {{ model_name }} | CV Accuracy: {{ cv_acc }}% | Test Accuracy: {{ test_acc }}%
        </div>
    </div>

    <div class="container">
        <div>
            <div class="card">
                <h2>📊 Thống kê Hệ thống</h2>
                <div class="stats-grid">
                    <div class="stat-box">
                        <div class="stat-value">{{ test_acc }}%</div>
                        <div class="stat-label">Test Accuracy</div>
                    </div>
                    <div class="stat-box">
                        <div class="stat-value">{{ n_features }}</div>
                        <div class="stat-label">Đặc trưng</div>
                    </div>
                    <div class="stat-box">
                        <div class="stat-value">{{ f1_score }}%</div>
                        <div class="stat-label">F1-Score</div>
                    </div>
                    <div class="stat-box">
                        <div class="stat-value" id="totalPredictions">0</div>
                        <div class="stat-label">Dự đoán</div>
                    </div>
                </div>
                <div style="background:#f8f9ff;border-radius:10px;padding:15px;font-size:0.9em;color:#555;">
                    <strong>🔬 D-DOA Params:</strong> C={{ best_c }} | γ={{ best_gamma }} | AUC={{ auc_val }}
                </div>
            </div>

            <div class="card" style="max-height: 680px; overflow-y: auto; margin-top: 25px;">
                <h2>📝 Nhập Thông tin Bệnh nhân</h2>
                <form id="predictionForm">
                    <div class="form-group"><label>👤 Họ và Tên</label><input type="text" name="patient_name" placeholder="VD: Nguyễn Văn A" required></div>
                    <div class="form-group"><label>🎂 Tuổi</label><input type="number" name="age" min="20" max="100" value="50" required></div>
                    <div class="form-group"><label>⚥ Giới tính</label><select name="sex" required><option value="1">Nam</option><option value="0">Nữ</option></select></div>
                    <div class="form-group"><label>📏 Chiều cao (cm)</label><input type="number" name="height" min="100" max="250" value="170" required></div>
                    <div class="form-group"><label>⚖️ Cân nặng (kg)</label><input type="number" name="weight" min="30" max="200" value="70" required></div>
                    <div class="form-group"><label>💔 Loại đau ngực</label><select name="cp" required><option value="1">Typical Angina</option><option value="2">Atypical Angina</option><option value="3">Non-anginal</option><option value="4">Asymptomatic</option></select></div>
                    <div class="form-group"><label>🩺 Huyết áp (mmHg)</label><input type="number" name="trestbps" min="80" max="220" value="120" required></div>
                    <div class="form-group"><label>🧪 Cholesterol (mg/dl)</label><input type="number" name="chol" min="100" max="600" value="200" required></div>
                    <div class="form-group"><label>🍬 Đường huyết đói &gt; 120</label><select name="fbs" required><option value="0">Không</option><option value="1">Có</option></select></div>
                    <div class="form-group"><label>📈 Kết quả ECG</label><select name="restecg" required><option value="0">Normal</option><option value="1">ST-T Abnormality</option><option value="2">LV Hypertrophy</option></select></div>
                    <div class="form-group"><label>💗 Nhịp tim tối đa (bpm)</label><input type="number" name="thalach" min="60" max="220" value="150" required></div>
                    <div class="form-group"><label>🏃 Đau ngực khi vận động</label><select name="exang" required><option value="0">Không</option><option value="1">Có</option></select></div>
                    <div class="form-group"><label>📉 ST Depression</label><input type="number" name="oldpeak" min="0" max="10" step="0.1" value="1.0" required></div>
                    <div class="form-group"><label>📊 Độ dốc ST</label><select name="slope" required><option value="1">Upsloping</option><option value="2">Flat</option><option value="3">Downsloping</option></select></div>
                    <div class="form-group"><label>🔬 Số mạch máu chính</label><select name="ca" required><option value="0">0</option><option value="1">1</option><option value="2">2</option><option value="3">3</option></select></div>
                    <div class="form-group"><label>🧬 Thalassemia</label><select name="thal" required><option value="3">Normal</option><option value="6">Fixed Defect</option><option value="7">Reversible Defect</option></select></div>
                    <div class="btn-group">
                        <button type="submit" class="btn-primary">🔍 Phân tích</button>
                        <button type="reset" class="btn-secondary">🔄 Reset</button>
                    </div>
                </form>
            </div>
        </div>

        <div>
            <div class="card">
                <h2>📋 Kết quả Dự đoán</h2>
                <div class="loading" id="loading">
                    <div class="spinner"></div>
                    <p style="color: #667eea; font-size: 1.3em; font-weight: 600;">Đang phân tích dữ liệu...</p>
                </div>
                <div class="report-container" id="reportContainer">
                    <p style="text-align: center; color: #999; padding: 50px;">Vui lòng nhập thông tin bệnh nhân và bấm "Phân tích"</p>
                </div>
            </div>

            <div class="card" style="margin-top: 25px;">
                <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 20px;">
                    <h2 style="margin: 0;">📜 Lịch sử Dự đoán</h2>
                    <button onclick="downloadCSV()" class="btn-success" style="flex: none; padding: 10px 20px;">💾 Tải CSV</button>
                </div>
                <div class="history-container" id="historyContainer">
                    <p style="text-align: center; color: #999; padding: 30px;">Chưa có lịch sử dự đoán</p>
                </div>
            </div>
        </div>
    </div>

    <script>
        window.addEventListener(\'load\', () => { loadHistory(); });

        document.getElementById(\'predictionForm\').addEventListener(\'submit\', async (e) => {
            e.preventDefault();
            const formData = new FormData(e.target);
            const data = {};
            for (let [key, value] of formData.entries()) {
                data[key] = key === \'patient_name\' ? value : parseFloat(value);
            }
            document.getElementById(\'loading\').classList.add(\'show\');
            document.getElementById(\'reportContainer\').innerHTML = \'\';
            try {
                const response = await fetch(\'/predict\', {
                    method: \'POST\',
                    headers: { \'Content-Type\': \'application/json\' },
                    body: JSON.stringify(data)
                });
                const result = await response.json();
                document.getElementById(\'loading\').classList.remove(\'show\');
                if (result.success) {
                    displayResult(result.result, data);
                    loadHistory();
                    const currentCount = parseInt(document.getElementById(\'totalPredictions\').textContent);
                    document.getElementById(\'totalPredictions\').textContent = currentCount + 1;
                } else { alert(\'Lỗi: \' + result.error); }
            } catch (error) {
                document.getElementById(\'loading\').classList.remove(\'show\');
                alert(\'Lỗi kết nối: \' + error.message);
            }
        });

        function displayResult(result, patientData) {
            const isHealthy = result.prediction === 0;
            const statusText = isHealthy ? \'✅ KHỎE MẠNH\' : \'⚠️ CÓ KHẢ NĂNG BỊ BỆNH TIM\';
            const statusIcon = isHealthy ? \'💚\' : \'❤️\';
            const height = parseFloat(patientData.height) / 100;
            const weight = parseFloat(patientData.weight);
            const bmi = (weight / (height * height)).toFixed(1);
            let bmiCategory = bmi < 18.5 ? \'Thiếu cân\' : bmi < 25 ? \'Bình thường ✓\' : bmi < 30 ? \'Thừa cân\' : \'Béo phì\';
            const heartColor = isHealthy ? \'#2ecc71\' : \'#e74c3c\';
            const heartStroke = isHealthy ? \'#27ae60\' : \'#c0392b\';
            const heartClass = isHealthy ? \'heart-healthy\' : \'heart-risk\';

            const html = `
                <div style="text-align:center;padding:30px;background:linear-gradient(135deg,#f8f9ff,#fff5f8);border-radius:15px;margin:20px 0;">
                    <div style="font-size:4em;margin-bottom:15px;">${statusIcon}</div>
                    <h3 style="font-size:1.8em;color:#667eea;margin-bottom:15px;">${patientData.patient_name}</h3>
                    <div style="display:inline-block;padding:15px 40px;border-radius:30px;font-size:1.4em;font-weight:bold;margin:15px 0;background:${isHealthy ? \'linear-gradient(135deg,#2ecc71,#27ae60)\' : \'linear-gradient(135deg,#e74c3c,#c0392b)\'};color:white;">${statusText}</div>
                    <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:15px;margin-top:25px;">
                        <div style="background:white;padding:15px;border-radius:12px;"><div style="color:#666;font-size:0.9em;">Tuổi</div><div style="font-size:1.5em;font-weight:bold;color:#333;">${patientData.age}</div></div>
                        <div style="background:white;padding:15px;border-radius:12px;"><div style="color:#666;font-size:0.9em;">Giới tính</div><div style="font-size:1.5em;font-weight:bold;color:#333;">${patientData.sex==1?\'Nam\':\'Nữ\'}</div></div>
                        <div style="background:white;padding:15px;border-radius:12px;"><div style="color:#666;font-size:0.9em;">BMI</div><div style="font-size:1.5em;font-weight:bold;color:#333;">${bmi}</div></div>
                        <div style="background:white;padding:15px;border-radius:12px;"><div style="color:#666;font-size:0.9em;">Huyết áp</div><div style="font-size:1.5em;font-weight:bold;color:#333;">${patientData.trestbps}</div></div>
                    </div>
                </div>

                <div class="body-section">
                    <div class="body-container">
                        <h4>🧍 Mô phỏng Cơ thể 2D</h4>
                        <svg class="human-body-svg" viewBox="0 0 200 400" xmlns="http://www.w3.org/2000/svg">
                            <circle cx="100" cy="30" r="25" fill="#ffd7a8" stroke="#333" stroke-width="2"/>
                            <circle cx="92" cy="25" r="3" fill="#333"/>
                            <circle cx="108" cy="25" r="3" fill="#333"/>
                            <path d="M 90 38 Q 100 42 110 38" stroke="#333" stroke-width="2" fill="none"/>
                            <rect x="95" y="52" width="10" height="15" fill="#ffd7a8" stroke="#333" stroke-width="1"/>
                            <ellipse cx="100" cy="130" rx="50" ry="80" fill="#e8f4f8" stroke="#333" stroke-width="2"/>
                            <g class="${heartClass}">
                                <path d="M 100 110 C 100 100, 90 95, 85 100 C 80 105, 80 110, 85 115 L 100 130 L 115 115 C 120 110, 120 105, 115 100 C 110 95, 100 100, 100 110 Z" fill="${heartColor}" stroke="${heartStroke}" stroke-width="2"/>
                                <circle cx="100" cy="110" r="3" fill="#fff" opacity="0.8"/>
                            </g>
                            <ellipse cx="75" cy="120" rx="18" ry="30" fill="#ffc8dd" stroke="#666" stroke-width="1" opacity="0.6"/>
                            <ellipse cx="125" cy="120" rx="18" ry="30" fill="#ffc8dd" stroke="#666" stroke-width="1" opacity="0.6"/>
                            <rect x="40" y="80" width="15" height="100" rx="7" fill="#ffd7a8" stroke="#333" stroke-width="2"/>
                            <rect x="145" y="80" width="15" height="100" rx="7" fill="#ffd7a8" stroke="#333" stroke-width="2"/>
                            <circle cx="47" cy="185" r="10" fill="#ffd7a8" stroke="#333" stroke-width="2"/>
                            <circle cx="153" cy="185" r="10" fill="#ffd7a8" stroke="#333" stroke-width="2"/>
                            <rect x="75" y="200" width="18" height="140" rx="9" fill="#a8d8ea" stroke="#333" stroke-width="2"/>
                            <rect x="107" y="200" width="18" height="140" rx="9" fill="#a8d8ea" stroke="#333" stroke-width="2"/>
                            <ellipse cx="84" cy="350" rx="12" ry="8" fill="#333"/>
                            <ellipse cx="116" cy="350" rx="12" ry="8" fill="#333"/>
                            <text x="100" y="260" text-anchor="middle" font-size="14" fill="#667eea" font-weight="bold">BMI: ${bmi}</text>
                            <text x="100" y="278" text-anchor="middle" font-size="11" fill="#666">${bmiCategory}</text>
                        </svg>
                        <div class="body-stats">
                            <div style="font-weight:bold;color:#667eea;margin-bottom:12px;text-align:center;">Chỉ số Cơ thể</div>
                            <div class="body-stats-grid">
                                <div class="body-stats-item"><strong>Chiều cao:</strong> ${patientData.height} cm</div>
                                <div class="body-stats-item"><strong>Cân nặng:</strong> ${patientData.weight} kg</div>
                                <div class="body-stats-item"><strong>BMI:</strong> ${bmi} kg/m²</div>
                                <div class="body-stats-item"><strong>Phân loại:</strong> ${bmiCategory}</div>
                            </div>
                        </div>
                    </div>
                    <div>
                        <h4 style="color:#667eea;margin-bottom:20px;font-size:1.4em;">📊 Các Chỉ số Tim mạch</h4>
                        <div class="metrics-container">
                            ${getMetricBox(\'🩺\',\'Huyết áp\',patientData.trestbps,\'mmHg\',patientData.trestbps)}
                            ${getMetricBox(\'🧪\',\'Cholesterol\',patientData.chol,\'mg/dl\',patientData.chol)}
                            ${getMetricBox(\'💗\',\'Nhịp tim tối đa\',patientData.thalach,\'bpm\',patientData.thalach)}
                            ${getMetricBox(\'🍬\',\'Đường huyết\',patientData.fbs==1?\'>120\':\'<120\',\'mg/dl\',patientData.fbs)}
                            ${getECGMetricBox(patientData.restecg)}
                            ${getVesselMetricBox(patientData.ca)}
                        </div>
                    </div>
                </div>

                <div style="display:grid;grid-template-columns:repeat(2,1fr);gap:20px;margin:30px 0;max-width:800px;margin-left:auto;margin-right:auto;">
                    <div style="background:linear-gradient(135deg,#2ecc71,#27ae60);color:white;padding:30px;border-radius:15px;text-align:center;box-shadow:0 8px 25px rgba(46,204,113,0.3);">
                        <div style="font-size:1.1em;margin-bottom:15px;opacity:0.9;">🟢 Khả năng không bị bệnh tim</div>
                        <div style="font-size:3em;font-weight:bold;">${result.healthy_probability.toFixed(1)}%</div>
                    </div>
                    <div style="background:linear-gradient(135deg,#e74c3c,#c0392b);color:white;padding:30px;border-radius:15px;text-align:center;box-shadow:0 8px 25px rgba(231,76,60,0.3);">
                        <div style="font-size:1.1em;margin-bottom:15px;opacity:0.9;">🔴 Khả năng bị bệnh tim</div>
                        <div style="font-size:3em;font-weight:bold;">${result.disease_probability.toFixed(1)}%</div>
                    </div>
                </div>

                ${getRecommendations(isHealthy, patientData, bmi)}

                <div style="background:${isHealthy?\'#d4edda\':\'#f8d7da\'};border:2px solid ${isHealthy?\'#28a745\':\'#dc3545\'};border-radius:15px;padding:20px;margin:20px 0;">
                    <p style="color:${isHealthy?\'#155724\':\'#721c24\'};font-size:1.05em;line-height:1.6;text-align:center;">
                        ${isHealthy ? \'✅ Kết quả cho thấy bạn có nguy cơ thấp mắc bệnh tim. Tuy nhiên, hãy duy trì lối sống lành mạnh và kiểm tra sức khỏe định kỳ.\' : \'⚠️ Kết quả cho thấy có dấu hiệu nguy cơ bệnh tim. Khuyến nghị gặp bác sĩ chuyên khoa tim mạch để được tư vấn và kiểm tra chi tiết.\'}
                    </p>
                </div>
                <div style="text-align:center;margin-top:25px;padding-top:20px;border-top:1px solid #e0e0e0;">
                    <p style="color:#666;font-size:0.95em;">Model: <strong>${result.model_name}</strong> | Thời gian: <strong>${result.timestamp}</strong></p>
                </div>
            `;
            document.getElementById(\'reportContainer\').innerHTML = html;
            document.getElementById(\'reportContainer\').classList.add(\'show\');
        }

        function getMetricBox(icon, label, value, unit, rawValue) {
            let status = \'normal\', statusText = \'Bình thường\';
            if (label === \'Huyết áp\') {
                if (rawValue >= 140) { status = \'danger\'; statusText = \'Rất cao\'; }
                else if (rawValue >= 120) { status = \'warning\'; statusText = \'Cao\'; }
                else { status = \'normal\'; statusText = \'Tốt\'; }
            } else if (label === \'Cholesterol\') {
                if (rawValue >= 240) { status = \'danger\'; statusText = \'Rất cao\'; }
                else if (rawValue >= 200) { status = \'warning\'; statusText = \'Cao\'; }
                else { status = \'normal\'; statusText = \'Tốt\'; }
            } else if (label === \'Nhịp tim tối đa\') {
                status = rawValue >= 100 ? \'normal\' : \'warning\';
                statusText = rawValue >= 100 ? \'Tốt\' : \'Thấp\';
            } else if (label === \'Đường huyết\') {
                status = rawValue == 0 ? \'normal\' : \'warning\';
                statusText = rawValue == 0 ? \'Bình thường\' : \'Cao\';
            }
            return `<div class="metric-box ${status}"><div class="metric-icon">${icon}</div><div class="metric-label">${label}</div><div class="metric-value">${value}<span class="metric-unit">${unit}</span></div><span class="metric-status status-${status}">${statusText}</span></div>`;
        }

        function getECGMetricBox(ecgValue) {
            const ecgTexts = [\'Normal\', \'ST-T Abnormal\', \'LV Hypertrophy\'];
            const status = ecgValue == 0 ? \'normal\' : \'warning\';
            const statusText = ecgValue == 0 ? \'Bình thường\' : \'Bất thường\';
            return `<div class="metric-box ${status}"><div class="metric-icon">📈</div><div class="metric-label">Kết quả ECG</div><div class="metric-value" style="font-size:1.6em;">${ecgTexts[ecgValue]}</div><span class="metric-status status-${status}">${statusText}</span></div>`;
        }

        function getVesselMetricBox(vessels) {
            let status = \'normal\', statusText = \'Tốt\';
            if (vessels >= 2) { status = \'danger\'; statusText = \'Hẹp nghiêm trọng\'; }
            else if (vessels >= 1) { status = \'warning\'; statusText = \'Có hẹp\'; }
            return `<div class="metric-box ${status}"><div class="metric-icon">🔬</div><div class="metric-label">Mạch máu chính</div><div class="metric-value">${vessels}<span class="metric-unit">bị hẹp</span></div><span class="metric-status status-${status}">${statusText}</span></div>`;
        }

        function getRecommendations(isHealthy, data, bmi) {
            let recommendations = [];
            if (isHealthy) {
                recommendations.push({ icon:\'🏃\', text:\'<strong>Vận động thể chất (AHA):</strong> Ít nhất 150 phút/tuần hoạt động cường độ vừa HOẶC 75 phút/tuần hoạt động cường độ cao. Kết hợp bài tập tăng cường cơ bắp 2 ngày/tuần.\', source:\'American Heart Association\', url:\'https://www.heart.org/en/healthy-living/fitness/fitness-basics/aha-recs-for-physical-activity-in-adults\' });
                recommendations.push({ icon:\'🥗\', text:\'<strong>Chế độ ăn DASH (AHA/ACC):</strong> Ăn nhiều rau củ đa dạng, trái cây, nguyên hạt. Hạn chế muối <2,300mg/ngày, đường thêm vào <6%, chất béo bão hòa <10% tổng năng lượng.\', source:\'AHA/ACC Guidelines 2019\', url:\'https://millionhearts.hhs.gov/data-reports/factsheets/ABCS.html\' });
                recommendations.push({ icon:\'🚭\', text:\'<strong>Không hút thuốc (WHO):</strong> Tránh hoàn toàn thuốc lá và khói thuốc thụ động. Hút thuốc là yếu tố nguy cơ hàng đầu gây bệnh tim mạch.\', source:\'WHO Cardiovascular Guidelines\', url:\'https://www.who.int/health-topics/cardiovascular-diseases\' });
                recommendations.push({ icon:\'⚖️\', text:\'<strong>Duy trì cân nặng khỏe mạnh (CDC):</strong> BMI lý tưởng 18.5-24.9 kg/m². Vòng eo <102cm (nam) hoặc <88cm (nữ) để giảm nguy cơ tim mạch.\', source:\'CDC Heart Disease Prevention\', url:\'https://www.cdc.gov/heart-disease/prevention/\' });
                recommendations.push({ icon:\'🩺\', text:\'<strong>Kiểm tra định kỳ (ACC/AHA):</strong> Đo huyết áp, cholesterol, đường huyết hàng năm. Đánh giá nguy cơ tim mạch 10 năm với bác sĩ để can thiệp sớm.\', source:\'ACC/AHA Prevention Guidelines\', url:\'https://www.acc.org/Guidelines\' });
                recommendations.push({ icon:\'😌\', text:\'<strong>Quản lý căng thẳng (AHA):</strong> Thực hành kỹ thuật giảm stress: thiền, yoga, hít thở sâu. Ngủ đủ 7-9 giờ/đêm. Stress mãn tính làm tăng nguy cơ tim mạch.\', source:\'AHA Stress Management\', url:\'https://www.heart.org/en/healthy-living/healthy-lifestyle/lifes-essential-8\' });
            } else {
                recommendations.push({ icon:\'🚨\', text:\'<strong>KHẨN CẤP - GẶP BÁC SĨ TIM MẠCH NGAY:</strong> Kết quả cho thấy nguy cơ bệnh tim. Cần thăm khám chuyên khoa tim mạch càng sớm càng tốt. Đây KHÔNG thay thế chẩn đoán y khoa.\', source:\'American Heart Association\', url:\'https://www.heart.org/en/health-topics/heart-attack\' });
                recommendations.push({ icon:\'🔬\', text:\'<strong>Xét nghiệm chuyên sâu cần làm:</strong> ECG, Echo tim (siêu âm tim), Stress test, xét nghiệm lipid máu chi tiết, HbA1c, CRP (viêm), có thể cần CT/MRI tim theo chỉ định bác sĩ.\', source:\'ACC/AHA Diagnostic Guidelines\', url:\'https://www.acc.org/Guidelines\' });
                recommendations.push({ icon:\'💊\', text:\'<strong>Tuân thủ điều trị nghiêm ngặt:</strong> Nếu được kê đơn thuốc (statin, thuốc huyết áp...), uống đúng liều, đúng giờ. TUYỆT ĐỐI không tự ý ngừng thuốc.\', source:\'WHO Medication Adherence\', url:\'https://www.who.int/news-room/fact-sheets/detail/cardiovascular-diseases-(cvds)\' });
                recommendations.push({ icon:\'🚫\', text:\'<strong>Loại bỏ yếu tố nguy cơ ngay:</strong> BỎ THUỐC LÁ ngay lập tức, tránh rượu bia, hạn chế caffeine, tránh thức ăn nhiều muối/mỡ/đường.\', source:\'AHA Risk Factor Management\', url:\'https://www.heart.org/en/healthy-living\' });
                recommendations.push({ icon:\'🏥\', text:\'<strong>Nhận biết dấu hiệu cảnh báo:</strong> Đau ngực, khó thở, đau lan ra cánh tay/vai/cổ/hàm, mệt mỏi bất thường → GỌI CẤP CỨU 115 NGAY.\', source:\'Emergency Cardiovascular Care\', url:\'https://www.heart.org/en/health-topics/heart-attack/warning-signs-of-a-heart-attack\' });
            }
            if (data.trestbps >= 140) recommendations.push({ icon:\'⚠️\', text:\'<strong>Huyết áp cao ≥140 mmHg:</strong> Giảm muối <1,500mg/ngày, tăng kali, giảm cân, tập thể dục đều đặn. Có thể cần thuốc hạ huyết áp - hỏi bác sĩ.\', source:\'AHA Hypertension Guidelines\', url:\'https://www.heart.org/en/health-topics/high-blood-pressure\' });
            if (data.chol >= 240) recommendations.push({ icon:\'🧪\', text:\'<strong>Cholesterol rất cao ≥240 mg/dL:</strong> Ăn ít chất béo bão hòa, tránh trans fat, tăng chất xơ hòa tan (yến mạch, đậu). Có thể cần statin - hỏi bác sĩ.\', source:\'ACC/AHA Cholesterol Guidelines\', url:\'https://www.ahajournals.org/doi/10.1161/CIR.0000000000000625\' });
            if (bmi >= 25) { const bmiStatus = bmi >= 30 ? \'béo phì\' : \'thừa cân\'; const targetWeight = (24.9 * Math.pow(data.height/100, 2)).toFixed(1); recommendations.push({ icon:\'⚖️\', text:`<strong>BMI ${bmi} (${bmiStatus}):</strong> Mục tiêu giảm xuống BMI <25 (cân nặng ~${targetWeight}kg). Giảm 0.5-1kg/tuần. Tham khảo chuyên gia dinh dưỡng.`, source:\'CDC Healthy Weight\', url:\'https://www.cdc.gov/healthyweight/\' }); }

            let html = \'<div class="recommendations-section"><h4>💡 KHUYẾN NGHỊ Y TẾ DỰA TRÊN BẰNG CHỨNG KHOA HỌC</h4>\';
            html += \'<p style="text-align:center;color:#666;font-size:0.9em;margin-bottom:20px;font-style:italic;">Dựa trên hướng dẫn từ American Heart Association (AHA), American College of Cardiology (ACC), World Health Organization (WHO), Centers for Disease Control (CDC)</p>\';
            recommendations.forEach(rec => {
                html += `<div class="recommendation-item"><div class="recommendation-icon">${rec.icon}</div><div class="recommendation-text">${rec.text}<div style="margin-top:8px;font-size:0.85em;color:#888;font-style:italic;">📚 Nguồn: ${rec.source} ${rec.url ? `<a href="${rec.url}" target="_blank" style="color:#667eea;text-decoration:none;margin-left:5px;">🔗</a>` : \'\' }</div></div></div>`;
            });
            html += \'<div style="margin-top:25px;padding:20px;background:#fff3cd;border-radius:12px;border-left:5px solid #f39c12;"><p style="margin:0;color:#856404;font-size:0.95em;line-height:1.6;"><strong>⚠️ LƯU Ý QUAN TRỌNG:</strong> Đây là công cụ hỗ trợ sàng lọc, KHÔNG thay thế chẩn đoán y khoa. Kết quả chỉ mang tính tham khảo. Luôn tham khảo ý kiến bác sĩ chuyên khoa tim mạch.</p></div></div>\';
            return html;
        }

        async function loadHistory() {
            try {
                const response = await fetch(\'/history\');
                const data = await response.json();
                const container = document.getElementById(\'historyContainer\');
                if (data.success && data.history.length > 0) {
                    container.innerHTML = \'\';
                    data.history.slice().reverse().forEach(item => {
                        const isHealthy = item.prediction === 0;
                        const div = document.createElement(\'div\');
                        div.className = `history-item ${isHealthy ? \'healthy\' : \'risk\'}`;
                        div.innerHTML = `<div class="history-header"><div class="history-name">👤 ${item.patient_name}</div><div class="history-time">🕐 ${item.timestamp}</div></div><div class="history-result"><div class="history-status" style="color:${isHealthy?\'#2ecc71\':\'#e74c3c\'}">${isHealthy?\'✅ Khỏe mạnh\':\'⚠️ Nguy cơ bệnh tim\'}</div><div class="history-probabilities">🔴 Bệnh: <strong>${item.disease_probability.toFixed(1)}%</strong> | 🟢 Khỏe: <strong>${item.healthy_probability.toFixed(1)}%</strong> | 📊 Tin cậy: <strong>${item.confidence.toFixed(1)}%</strong></div></div>`;
                        container.appendChild(div);
                    });
                    document.getElementById(\'totalPredictions\').textContent = data.history.length;
                } else {
                    container.innerHTML = \'<p style="text-align:center;color:#999;padding:30px;">Chưa có lịch sử dự đoán</p>\';
                }
            } catch (error) { console.error(\'Lỗi load history:\', error); }
        }

        function downloadCSV() { window.location.href = \'/download_csv\'; }
    </script>
</body>
</html>
'''

print('✅ HTML template defined')

✅ HTML template defined


In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# 7C. FLASK ROUTES
# ─────────────────────────────────────────────────────────────────────────────
app = Flask(__name__)
CORS(app)

@app.route('/')
def index():
    return render_template_string(
        HTML_TEMPLATE,
        model_name=f'DOA-SVM',
        cv_acc=f'{best_cv*100:.1f}',
        test_acc=f'{acc*100:.1f}',
        f1_score=f'{f1*100:.1f}',
        n_features=n_feat,
        best_c=f'{best_C:.4f}',
        best_gamma=f'{best_gam:.6f}',
        auc_val=f'{auc:.4f}'
    )

@app.route('/predict', methods=['POST'])
def predict_api():
    try:
        data = request.json
        patient_name = data.get('patient_name', 'Unknown')
        feature_order = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
                        'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
        # Use only features in FEATURES list
        features = [data[f] for f in FEATURES]
        result = predict_heart_disease(features, patient_name, patient_data=data)
        return jsonify({'success': True, 'result': result})
    except Exception as e:
        return jsonify({'success': False, 'error': str(e)})

@app.route('/history', methods=['GET'])
def get_history():
    try:
        return jsonify({'success': True, 'history': prediction_history, 'total': len(prediction_history)})
    except Exception as e:
        return jsonify({'success': False, 'error': str(e), 'history': []})

@app.route('/download_csv', methods=['GET'])
def download_csv():
    try:
        if os.path.exists(CSV_FILE):
            return send_file(CSV_FILE, mimetype='text/csv', as_attachment=True,
                           download_name='heart_disease_predictions.csv')
        else:
            return jsonify({'success': False, 'error': 'File CSV không tồn tại'})
    except Exception as e:
        return jsonify({'success': False, 'error': str(e)})

print('✅ Flask routes defined')

✅ Flask routes defined


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 7D. LAUNCH FLASK + NGROK
# ─────────────────────────────────────────────────────────────────────────────
from pyngrok import ngrok

print("="*70)
print("🚀 KHỞI ĐỘNG WEB SERVER...")
print("="*70)
print(f"\n🏆 Model: DOA-SVM")
print(f"📊 CV Accuracy: {best_cv*100:.2f}%")
print(f"📊 Test Accuracy: {acc*100:.2f}%")
print(f"📈 AUC-ROC: {auc:.4f}")
print(f"📈 F1-Score: {f1:.4f}")
print(f"🔬 Best C: {best_C:.4f} | Best γ: {best_gam:.6f}")
print(f"⏱️  Runtime: {elapsed/60:.1f} min")

def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(3)
print("\n✅ Flask server started on port 5000")

print("\n" + "="*70)
print("🔐 THIẾT LẬP NGROK")
print("="*70)
ngrok_token = "3CFaqyavo72j2IqHC4eqsS4zpDj_2rvJvy25jK594jCSg9bQv"
print(f"\n🔑 Dùng token ngrok đã lưu sẵn: {ngrok_token}")

try:
    ngrok.set_auth_token(ngrok_token)
    public_url = ngrok.connect(5000)
    print("\n" + "="*70)
    print("✅ WEB APP ĐÃ KHỞI ĐỘNG THÀNH CÔNG!")
    print("="*70)
    print(f"\n🌍 LINK PUBLIC: {public_url}")
    print(f"🏠 Link Local : http://localhost:5000")
    print("\n" + "="*70)
except Exception as e:
    print(f"\n⚠️  Lỗi ngrok: {e}")
    print("📍 Chạy local: http://localhost:5000")

print("\n📝 Mọi dự đoán sẽ được tự động lưu vào CSV!")
print(f"📄 File CSV: {CSV_FILE}")
print("\n⏳ Server đang chạy... Nhấn Ctrl+C để dừng\n")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n\n🛑 Đang dừng server...")
    print("✅ Hoàn tất!")
    print("👋 Cảm ơn bạn đã sử dụng hệ thống!")

🚀 KHỞI ĐỘNG WEB SERVER...

🏆 Model: DOA-SVM
📊 CV Accuracy: 83.40%
📊 Test Accuracy: 81.37%
📈 AUC-ROC: 0.8854
📈 F1-Score: 0.8061
🔬 Best C: 0.9406 | Best γ: 0.131344
⏱️  Runtime: 35.7 min
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.152:5000
Press CTRL+C to quit



✅ Flask server started on port 5000

🔐 THIẾT LẬP NGROK

🔑 Dùng token ngrok đã lưu sẵn: 3CFaqyavo72j2IqHC4eqsS4zpDj_2rvJvy25jK594jCSg9bQv
⚠️  Lỗi ngrok: An error occurred while downloading ngrok from https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-darwin-arm64.zip: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)>
📍 Chạy local: http://localhost:5000

📝 Mọi dự đoán sẽ được tự động lưu vào CSV!
📄 File CSV: heart_disease_predictions.csv

⏳ Server đang chạy... Nhấn Ctrl+C để dừng

